In [43]:
import pandas as pd
import numpy as np

retenciones = pd.read_excel('retenciones.xlsx')
retenciones

,ClaveProdServ,Descripcion,Importe,ObjetoImp,Cantidad,ID_nodo_padre,Base,Impuesto,TipoFactor,TasaOCuota,UUID,Nombre del archivo,Emisor_Rfc,Emisor_Nombre,Comprobante_Fecha,Comprobante_TipoDeComprobante,Comprobante_Moneda
0,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,0,60082.0,2,Tasa,0.0400,BFBD99B0-13AB-4D7B-A742-6312339EBEA8,24-035408-L13106,TTM971210123,TRANSPORTES TERRESTRES MICHOACANOS,2024-12-07T17:57:49,I,MXN
1,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,0,60082.0,1,Tasa,0.0125,BFBD99B0-13AB-4D7B-A742-6312339EBEA8,24-035408-L13106,TTM971210123,TRANSPORTES TERRESTRES MICHOACANOS,2024-12-07T17:57:49,I,MXN
2,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,0,60082.0,2,Tasa,0.0400,ABCDEF12-13AB-4D7B-A742-6312339EBEA8,24-035408-L13106,TTM971210123,TRANSPORTES TERRESTRES MICHOACANOS,2024-12-07T17:57:49,I,MXN
3,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,0,60082.0,1,Tasa,0.0125,ABCDEF12-13AB-4D7B-A742-6312339EBEA8,24-035408-L13106,TTM971210123,TRANSPORTES TERRESTRES MICHOACANOS,2024-12-07T17:57:49,I,MXN


In [44]:
traslados = pd.read_excel('traslados.xlsx')
traslados

,ClaveProdServ,Descripcion,Importe,ObjetoImp,Cantidad,ID_nodo_padre,Base,Impuesto,TipoFactor,TasaOCuota,UUID,Nombre del archivo,Emisor_Rfc,Emisor_Nombre,Comprobante_Fecha,Comprobante_TipoDeComprobante,Comprobante_Moneda
0,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,0,60082.0,2,Tasa,0.16,BFBD99B0-13AB-4D7B-A742-6312339EBEA8,24-035408-L13106,TTM971210123,TRANSPORTES TERRESTRES MICHOACANOS,2024-12-07T17:57:49,I,MXN
1,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,0,60082.0,1,Tasa,0.04,ABCDEF12-13AB-4D7B-A742-6312339EBEA8,24-035408-L13106,TTM971210123,TRANSPORTES TERRESTRES MICHOACANOS,2024-12-07T17:57:49,I,MXN
2,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,1,60082.0,1,Tasa,0.04,ABCDEF12-13AB-4D7B-A742-6312339EBEA8,24-035408-L13106,TTM971210123,TRANSPORTES TERRESTRES MICHOACANOS,2024-12-07T17:57:49,I,MXN


In [33]:
CVES_IMPUESTO = {1: 'ISR', 2: 'IVA', 3: 'IEPS'}
COLS_ATTR_IMPUESTO = ['Base', 'TipoFactor','TasaOCuota']

group_by_cols = ['UUID', 'ID_nodo_padre']

def pivot_impuestos(impuestos:pd.DataFrame, group_by_cols:list[str]):
    """Genera un DataFrame con los impuestos pivoteados: 
    Cada impuesto se convierte en una columna con los atributos Base, TipoFactor, TasaOCuota, etc."""
    df = impuestos.copy()
    for clave,impuesto in CVES_IMPUESTO.items():
        for col in COLS_ATTR_IMPUESTO:
            df[f'{impuesto}_{col}'] = df[col].where(df['Impuesto'] == clave)

    df.drop(columns=COLS_ATTR_IMPUESTO, inplace=True)
    cols = df.columns.tolist()
    for col in group_by_cols:
        cols.remove(col)
    return df.groupby(group_by_cols)[cols].first().reset_index()

In [51]:
flattened_retenciones =pivot_impuestos(retenciones, group_by_cols)
flattened_traslados = pivot_impuestos(traslados, group_by_cols)

In [46]:
flattened_retenciones

,UUID,ID_nodo_padre,ClaveProdServ,Descripcion,Importe,ObjetoImp,Cantidad,Impuesto,Nombre del archivo,Emisor_Rfc,...,Comprobante_Moneda,ISR_Base,ISR_TipoFactor,ISR_TasaOCuota,IVA_Base,IVA_TipoFactor,IVA_TasaOCuota,IEPS_Base,IEPS_TipoFactor,IEPS_TasaOCuota
0,ABCDEF12-13AB-4D7B-A742-6312339EBEA8,0,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,2,24-035408-L13106,TTM971210123,...,MXN,60082.0,Tasa,0.0125,60082.0,Tasa,0.04,NaN,None,NaN
1,BFBD99B0-13AB-4D7B-A742-6312339EBEA8,0,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,2,24-035408-L13106,TTM971210123,...,MXN,60082.0,Tasa,0.0125,60082.0,Tasa,0.04,NaN,None,NaN


In [47]:
flattened_traslados

,UUID,ID_nodo_padre,ClaveProdServ,Descripcion,Importe,ObjetoImp,Cantidad,Impuesto,Nombre del archivo,Emisor_Rfc,...,Comprobante_Moneda,ISR_Base,ISR_TipoFactor,ISR_TasaOCuota,IVA_Base,IVA_TipoFactor,IVA_TasaOCuota,IEPS_Base,IEPS_TipoFactor,IEPS_TasaOCuota
0,ABCDEF12-13AB-4D7B-A742-6312339EBEA8,0,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,1,24-035408-L13106,TTM971210123,...,MXN,60082.0,Tasa,0.04,NaN,None,NaN,NaN,None,NaN
1,ABCDEF12-13AB-4D7B-A742-6312339EBEA8,1,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,1,24-035408-L13106,TTM971210123,...,MXN,60082.0,Tasa,0.04,NaN,None,NaN,NaN,None,NaN
2,BFBD99B0-13AB-4D7B-A742-6312339EBEA8,0,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,2,24-035408-L13106,TTM971210123,...,MXN,NaN,None,NaN,60082.0,Tasa,0.16,NaN,None,NaN


In [50]:
cols_ret = group_by_cols + [col for col in flattened_retenciones.columns if 'ISR' in col or 'IVA' in col or 'IEPS' in col]
flattened_traslados.merge(flattened_retenciones[cols_ret], on=group_by_cols, how='outer', suffixes=('_Tras', '_Ret'))

,UUID,ID_nodo_padre,ClaveProdServ,Descripcion,Importe,ObjetoImp,Cantidad,Impuesto,Nombre del archivo,Emisor_Rfc,...,IEPS_TasaOCuota_Tras,ISR_Base_Ret,ISR_TipoFactor_Ret,ISR_TasaOCuota_Ret,IVA_Base_Ret,IVA_TipoFactor_Ret,IVA_TasaOCuota_Ret,IEPS_Base_Ret,IEPS_TipoFactor_Ret,IEPS_TasaOCuota_Ret
0,ABCDEF12-13AB-4D7B-A742-6312339EBEA8,0,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,1,24-035408-L13106,TTM971210123,...,NaN,60082.0,Tasa,0.0125,60082.0,Tasa,0.04,NaN,None,NaN
1,ABCDEF12-13AB-4D7B-A742-6312339EBEA8,1,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,1,24-035408-L13106,TTM971210123,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BFBD99B0-13AB-4D7B-A742-6312339EBEA8,0,78101802,CONTENEDORES _x000D_\n[40HC (High Cube)] CAAU6...,60082.0,2,1,2,24-035408-L13106,TTM971210123,...,NaN,60082.0,Tasa,0.0125,60082.0,Tasa,0.04,NaN,None,NaN
